In [2]:
import pandas as pd

df_merged = pd.read_csv(r'C:\Users\Admin\Documents\GitHub\Horse\data_preprocessing\merged_data_kr.csv')
df_merged

,분할경주여부,마명,마번,기수번호,조교사번호,부담구분,출전번호,대상경주명,경주일자,경주거리,...,경주로상태,날씨,마체중,모마명,출생일,성별,소유자명,생산국,부마명,소재지
0,0,파이널축제,45339,080342,70170.0,2,3,일반,20230107,1300,...,양호,흐림,490.0,월드퀵,2020-03-10(6세),수,럭키,한국,지롤라모,서울경마공원
1,0,아르고리치,45921,080366,70146.0,2,6,일반,20230107,1300,...,양호,흐림,496.0,어너러리갤럽,2020-04-08(6세),수,우태율,한국,컬러즈플라잉,서울경마공원
2,0,파워에치드,45734,080339,70166.0,2,11,일반,20230107,1300,...,양호,흐림,483.0,베트파티,2020-03-18(6세),거,박정재,한국,에치드,서울경마공원
3,0,베스트선,45369,080405,70115.0,3,1,일반,20230107,1300,...,양호,흐림,479.0,아이월이프유월,2020-02-28(6세),거,베스트샤인조합,한국,테이크차지인디,서울경마공원
4,0,슈어윈,45240,080103,70096.0,3,7,일반,20230107,1300,...,양호,흐림,413.0,매그니피센트마인,2020-02-13(6세),암,지성배,한국,BERNARDINI,서울경마공원
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15943,0,스팔타,47422,80622,70164.0,3,7,일반,20260426,1200,...,건조,맑음,490.0,당대화신,2021-04-23(5세),거,김세권,한국,미스터크로우,서울경마공원
15944,0,원평리스트,47322,80588,70148.0,3,8,일반,20260426,1200,...,건조,맑음,469.0,시어리어슬리마인,2021-04-07(5세),암,김용재,한국,페더럴리스트,서울경마공원
15945,0,모어댄조크,51372,80550,70095.0,3,9,일반,20260426,1200,...,건조,맑음,471.0,마일드어뮤즈먼트,2022-04-01(4세),수,김정철,한국,PRACTICAL JOKE,서울경마공원
15946,0,제이에스로드,45729,80621,70179.0,3,10,일반,20260426,1200,...,건조,맑음,507.0,슬리프리스,2020-04-01(6세),수,박민지,한국,TWIRLING CANDY,서울경마공원


In [3]:
df_merged.columns

Index(['분할경주여부', '마명', '마번', '기수번호', '조교사번호', '부담구분', '출전번호', '대상경주명', '경주일자',
       '경주거리', '경주등급', '출전마구분', '경주번호', '야간경마여부', '순위', '확정배당율(단승식)', '편성두수',
       '마필등급', '경주기록', '출주두수', '경주로상태', '날씨', '마체중', '모마명', '출생일', '성별',
       '소유자명', '생산국', '부마명', '소재지'],
      dtype='str')

In [6]:
df_merged['편성두수'].isnull().sum()

np.int64(0)

In [8]:
df_merged['출주두수']

0        12.0
1        12.0
2        12.0
3        12.0
4        12.0
         ... 
15943    11.0
15944    11.0
15945    11.0
15946    11.0
15947    11.0
Name: 출주두수, Length: 15948, dtype: float64

In [36]:
cols_to_check = [
    '편성두수',     # rcPlansu
    '출주두수',     # rcVtdusu
    '마필등급',     # rcRank
    '경주기록',     # rcTime
    '경주로상태',   # track
    '날씨',         # weath
    '마체중'        # wgHr
]

result = []

for col in cols_to_check:
    total = len(df_merged)
    null_count = df_merged[col].isnull().sum()
    
    # 숫자형일 때만 0 체크
    if df_merged[col].dtype != 'object':
        zero_count = (df_merged[col] == 0).sum()
    else:
        zero_count = 0
    
    result.append({
        '컬럼명': col,
        '전체개수': total,
        '결측치개수': null_count,
        '결측치비율(%)': round(null_count / total * 100, 2),
        '0값개수': zero_count,
        '0값비율(%)': round(zero_count / total * 100, 2)
    })

df_check = pd.DataFrame(result)
print(df_check)

     컬럼명   전체개수  결측치개수  결측치비율(%)  0값개수  0값비율(%)
0   편성두수  15666      0       0.0     0      0.0
1   출주두수  15666      0       0.0     0      0.0
2   마필등급  15666      0       0.0     0      0.0
3   경주기록  15666      0       0.0     0      0.0
4  경주로상태  15666      0       0.0     0      0.0
5     날씨  15666      0       0.0     0      0.0
6    마체중  15666      0       0.0     0      0.0


In [ ]:
import numpy as np

# 0을 결측치로 바꿀 컬럼
zero_to_nan_cols = [
    '마필등급',
    '경주기록',
    '마체중'
]

# 0인 값들 NaN값으로 변환
df[zero_to_nan_cols] = df[zero_to_nan_cols].replace(0, np.nan)


# NaN데이터 드랍
df = df.dropna(subset=['경주기록'])
df = df.dropna(subset=['마필등급'])

In [19]:
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    return outliers, lower, upper

In [20]:
outliers_time, low, high = detect_outliers_iqr(df_merged, '경주기록')

print(f"범위: {low} ~ {high}")
print(outliers_time[['마번', '경주기록']].head())

범위: 39.949999999999996 ~ 138.75
         마번   경주기록
1427  45453  152.6
1776  45232  141.1
4283  53722  147.0
4284  43495  146.9
4769  54009  150.1


In [27]:
df_merged.loc[
    (df_merged['마체중'] < 350) | (df_merged['마체중'] > 650)
]

,분할경주여부,마명,마번,기수번호,조교사번호,부담구분,출전번호,대상경주명,경주일자,경주거리,...,경주로상태,날씨,마체중,모마명,출생일,성별,소유자명,생산국,부마명,소재지


In [28]:
df_merged[df_merged['출주두수'] > df_merged['편성두수']]

,분할경주여부,마명,마번,기수번호,조교사번호,부담구분,출전번호,대상경주명,경주일자,경주거리,...,경주로상태,날씨,마체중,모마명,출생일,성별,소유자명,생산국,부마명,소재지


In [24]:
print(df_merged["편성두수"].nunique())
print(df_merged["출주두수"].nunique())
print(df_merged["마필등급"].nunique())
print(df_merged["경주기록"].nunique())
print(df_merged["경주로상태"].nunique())
print(df_merged["날씨"].nunique())
print(df_merged["마체중"].nunique())

9
11
16
692
5
5
246


In [ ]:
df_merged = df_merged.dropna(subset=['경주기록'])


In [34]:
df_merged = df_merged.dropna(subset=['마필등급'])

In [35]:
df_merged['마필등급'].isnull().sum()

np.int64(0)

In [32]:
df_merged['경주기록'].isnull().sum()

np.int64(0)